In [ ]:
!pip install nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import nltk
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/ML/airline tweets dataset/airline_tweets.csv")

In [ ]:
data

In [ ]:
x = data["text"]
y = data["airline_sentiment"]

In [ ]:
classes = dict(zip(y.value_counts().keys(), range(3)))
classes

In [ ]:
y = y.map(classes).values
y

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1)
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 3), min_df=1)
hybrid_features = FeatureUnion([
    ('word_features', word_vectorizer),
    ('char_features', char_vectorizer)
])
x_train_vectorized = hybrid_features.fit_transform(x_train)
x_test_vectorized = hybrid_features.transform(x_test)

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
    )
lgbm_model.fit(x_train_vectorized, y_train)
y_train_pred = lgbm_model.predict(x_train_vectorized)
y_test_pred = lgbm_model.predict(x_test_vectorized)

In [ ]:
train_score_lgbm = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", train_score_lgbm)
test_score_lgbm = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_score_lgbm)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42
)
rf_model.fit(x_train_vectorized, y_train)
y_train_pred = rf_model.predict(x_train_vectorized)
y_test_pred = rf_model.predict(x_test_vectorized)

In [ ]:
train_score_rf = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", train_score_rf)
test_score_rf = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_score_rf)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    tree_method="hist",
    device="cuda",
    random_state=42

)
xgb_model.fit(x_train_vectorized, y_train)
y_train_pred = xgb_model.predict(x_train_vectorized)
y_test_pred = xgb_model.predict(x_test_vectorized)

In [ ]:
train_score_xg = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", train_score_xg)
test_score_xg = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_score_xg)